In [2]:
import os
import numpy as np
import pandas as pd
from datetime import datetime

# =========================================================
#  CONFIG — adjust paths to match your container mounts
# =========================================================
SOURCE_CSV   = "/home/jovyan/work/ntiProject/dataset.csv"
LANDING_ZONE = "/home/jovyan/work/ntiProject/data_raw_power_pings/"

os.makedirs(LANDING_ZONE, exist_ok=True)

BATCH_SIZE = 50   # 1000 rows / 50 = 20 batches total

# Dataset has no real reading date, so we simulate one per row — each
# record gets its own random date in this range instead of every row in
# a batch sharing the same ingestion date. This gives the fact table a
# realistic date distribution to analyze trends over time.
READING_DATE_START = "2024-01-01"
READING_DATE_END   = "2026-08-16"
RANDOM_SEED         = 42  # fixed seed -> reproducible dates across re-runs

# =========================================================
# SIMULATOR — Run ONCE to pre-generate all batches
# All columns are kept here — dropping/cleaning happens in transformation
# Airflow tracks batches by batch_file (deterministic name from run_number)
# =========================================================


def run_simulator():
    print("\n" + "=" * 60)
    print("  power consumption simulator:")
    print("  Splitting CSV into batches ==> landing zone")
    print("=" * 60 + "\n")

    try:
        df = pd.read_csv(SOURCE_CSV)
        df.columns = df.columns.str.strip()

        # "Outside Temperature" has a space in it — rename for downstream
        # Spark/SQL friendliness (matches how the bank pipeline avoided
        # spaced column names like "IP Address").
        df = df.rename(columns={"Outside Temperature": "OutsideTemperature"})

        # Dataset has no natural primary key — generate one from the
        # original row order so it's stable across re-runs/backfills.
        df = df.reset_index(drop=True)
        df.insert(0, "RecordID", df.index.map(lambda i: f"REC-{i:06d}"))

        # Simulate a per-row reading date (fixed seed -> same dates every
        # time this script runs, so re-running the simulator doesn't
        # shuffle history around).
        rng = np.random.default_rng(RANDOM_SEED)
        start_ts = pd.Timestamp(READING_DATE_START)
        end_ts   = pd.Timestamp(READING_DATE_END)
        day_span = (end_ts - start_ts).days
        random_offsets = rng.integers(0, day_span + 1, size=len(df))
        df["ReadingDate"] = (start_ts + pd.to_timedelta(random_offsets, unit="D")) \
            .strftime("%Y-%m-%d")

        total_rows    = len(df)
        total_batches = (total_rows + BATCH_SIZE - 1) // BATCH_SIZE

        print(f"  CSV Loaded Successfully")
        print(f"  Total rows    : {total_rows:,}")
        print(f"  Total columns : {len(df.columns)}")
        print(f"  Columns       : {list(df.columns)}")
        print(f"  Batch size    : {BATCH_SIZE} rows")
        print(f"  Total batches : {total_batches:,}")
        print(f"  Landing zone  : {LANDING_ZONE}\n")
        print("-" * 60)

    except Exception as e:
        print(f" Error loading CSV: {e}")
        return

    for batch_num in range(1, total_batches + 1):
        start = (batch_num - 1) * BATCH_SIZE
        end   = start + BATCH_SIZE

        chunk = df.iloc[start:end].copy()

        # Metadata columns
        chunk["source"]      = "power_consumption_sensor"
        chunk["ingested_at"] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        chunk["batch_id"]    = f"POWER-BATCH-{batch_num:04d}"

        # Save directly to landing zone (no subfolders)
        file_name = f"power_batch_{batch_num:04d}.json"
        out_path  = os.path.join(LANDING_ZONE, file_name)
        chunk.to_json(out_path, orient='records', lines=True)

        print(
            f"  Batch {batch_num:>5} / {total_batches} | "
            f"Rows: {start}-{min(end, total_rows) - 1:<6} | "
            f"Records: {len(chunk):<3} | "
            f"File: {file_name} | Time: {chunk['ingested_at'].iloc[0]} "
        )

    print(f"\n{'=' * 60}")
    print(f"  SIMULATION COMPLETE")
    print(f"  {total_batches:,} batch files saved in landing zone")
    print(f"  Next step -> Trigger Airflow DAG: power_etl_pipeline")
    print(f"{'=' * 60}\n")


if __name__ == "__main__":
    try:
        run_simulator()
    except KeyboardInterrupt:
        print("\n  Simulator stopped by user.")


  power consumption simulator:
  Splitting CSV into batches ==> landing zone

  CSV Loaded Successfully
  Total rows    : 1,000
  Total columns : 10
  Columns       : ['RecordID', 'RoomArea', 'NumberofAppliances', 'OutsideTemperature', 'InsulationThickness', 'BuildingType', 'HVACSystem', 'AverageTemperature', 'EnergyConsumption', 'ReadingDate']
  Batch size    : 50 rows
  Total batches : 20
  Landing zone  : /home/jovyan/work/ntiProject/data_raw_power_pings/

------------------------------------------------------------
  Batch     1 / 20 | Rows: 0-49     | Records: 50  | File: power_batch_0001.json | Time: 2026-08-17 12:36:33 
  Batch     2 / 20 | Rows: 50-99     | Records: 50  | File: power_batch_0002.json | Time: 2026-08-17 12:36:33 
  Batch     3 / 20 | Rows: 100-149    | Records: 50  | File: power_batch_0003.json | Time: 2026-08-17 12:36:33 
  Batch     4 / 20 | Rows: 150-199    | Records: 50  | File: power_batch_0004.json | Time: 2026-08-17 12:36:33 
  Batch     5 / 20 | Rows: 20